LangChain Essentials

Until 2021, to use an AI model for a specific use-case we would need to fine-tune the model weights themselves. That would require huge amounts of training data and significant compute to fine-tune any reasonably performing model.

Instruction fine-tuned Large Language Models (LLMs) changed this fundamental rule of applying AI models to new use-cases. Rather than needing to either train a model from scratch or fine-tune an existing model, these new LLMs could adapt incredibly well to a new problem or use-case with nothing more than a prompt change.

Prompts allow us to completely change the functionality of an AI pipeline. Through natural language we simply tell our LLM what it needs to do, and with the right AI pipeline and prompting, it often works.

LangChain naturally has many functionalities geared towards helping us build our prompts. We can build very dynamic prompting pipelines that modifying the structure and content of what we feed into our LLM based on essentially any parameter we would like. In this example, we'll explore the essentials to prompting in LangChain and apply this in a demo Retrieval Augmented Generation (RAG) pipeline.

Importing the libraries and model

In [1]:
import langchain_core
import langchain_community
import langchain_ollama
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

c:\Users\PatelDharmikkumar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Basic Prompting
We'll start by looking at the various parts of our prompt. For RAG use-cases we'll typically have three core components however this is very use-cases dependant and can vary significantly. Nonetheless, for RAG we will typically see:

Rules for our LLM: this part of the prompt sets up the behavior of our LLM, how it should approach responding to user queries, and simply providing as much information as possible about what we're wanting to do as possible. We typically place this within the system prompt of an chat LLM.

Context: this part is RAG-specific. The context refers to some external information that we may have retrieved from a web search, database query, or often a vector database. This external information is the Retrieval Augmentation part of RAG. For chat LLMs we'll typically place this inside the chat messages between the assistant and user.

Question: this is the input from our user. In the vast majority of cases the question/query/user input will always be provided to the LLM (and typically through a user message). However, the format and location of this being provided often changes.

Answer: this is the answer from our assistant, again this is very typical and we'd expect this with every use-case.

The below is an example of how a RAG prompt may look:

Answer the question based on the context below,                 }
if you cannot answer the question using the                     }--->  (Rules) For Our Prompt
provided information answer with "I don't know"                 }

Context: Aurelio AI is an AI development studio                 }
focused on the fields of Natural Language Processing (NLP)      }
and information retrieval using modern tooling                  }--->   Context AI has
such as Large Language Models (LLMs),                           }
vector databases, and LangChain.                                }

Question: Does Aurelio AI do anything related to LangChain?     }--->   User Question

Answer:                                                         }--->   AI Answer
Here we can see how the AI will appoach our question, as you can see we have a formulated response, if the context has the answer, then use the context to answer the question, if not, say I don't know, then we also have context and question which are being passed into this similarly to paramaters in a function.

In [2]:
prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Context: {context}
"""

LangChain uses a ChatPromptTemplate object to format the various prompt types into a single list which will be passed to our LLM:

In [3]:
from langchain_core.prompts import ChatPromptTemplate

# passing the template to the LangChain model
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system",prompt),#the context of the prompt is given the system role,we haven't explicitly defined the system role but it is used to give the context of the prompt
        ("user","{query}")
    ]#giving the role of the message and the variable to be replaced in the prompt
)

When we call the template it will expect us to provide two variables, the context and the query. Both of these variables are pulled from the strings we wrote, as LangChain interprets curly-bracket syntax (ie {context} and {query}) as indicating a dynamic variable that we expect to be inserted at query time. We can see that these variables have been picked up by our template object by viewing it's input_variables attribute

In [4]:
prompt_template.input_variables

['context', 'query']

We can also view the structure of the messages (currently prompt templates) that the ChatPromptTemplate will construct by viewing the messages attribute:

In [5]:
prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]

From this, we can see that each tuple provided when using ChatPromptTemplate.from_messages becomes an individual prompt template itself. Within each of these tuples, the first value defines the role of the message, which is typically system, human, or ai. Using these tuples is shorthand for the following, more explicit code

In [6]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(prompt),#the system message prompt template is used to give the context of the prompt and the template is passed as an argument to it
    HumanMessagePromptTemplate.from_template("{query}")#importing the human message prompt template and giving the template for the human message
])

We can see the structure of this new chat prompt template is identical to our previous:

In [7]:
prompt_template

ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

 **Invoking our LLM with Templates**

We've defined our prompt template, now let's define out LLM and run it with our template and a user query.

In [8]:
pipeline = (
    {#giving input variables to the pipeline, these variables will be used in the prompt template,we can change their names as we want but they should match the variable names in the prompt template
        "query" : lambda x : x["query"], #lamda function is used to extract the query from the input and pass it to the prompt template
        "context" : lambda x : x["context"] # lamda function is used to extract the context from the input and pass it to the prompt template
    }
    | prompt_template
    | llm
)

Creating Context and Query

In [9]:
context = """Aurelio AI is an AI company developing tooling for AI
engineers. Their focus is on language AI with the team having strong
expertise in building AI agents and a strong background in
information retrieval.

The company is behind several open source frameworks, most notably
Semantic Router and Semantic Chunkers. They also have an AI
Platform providing engineers with tooling to help them build with
AI. Finally, the team also provides development services to other
organizations to help them bring their AI tech to market.

Aurelio AI became LangChain Experts in September 2024 after a long
track record of delivering AI solutions built with the LangChain
ecosystem."""

query = "what does Aurelio AI do?"

In [10]:
pipeline.invoke({"query": query, "context" : context})

AIMessage(content='Aurelio AI is an AI company that develops tooling for AI engineers, specifically focusing on language AI. They have expertise in building AI agents and information retrieval. Their work includes:\n\n1. Developing open-source frameworks (notably Semantic Router and Semantic Chunkers)\n2. Creating an AI Platform to help engineers build with AI\n3. Providing development services to other organizations to bring their AI tech to market\n\nThey also became LangChain Experts in September 2024, indicating a strong track record of delivering AI solutions built using the LangChain ecosystem.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-19T12:47:38.6318155Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16659031400, 'load_duration': 204790700, 'prompt_eval_count': 191, 'prompt_eval_duration': 462725700, 'eval_count': 111, 'eval_duration': 15884242400, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id=

Our LLM pipeline is able to consume the information from the context and use it to answer the user's query. Ofcourse, we would not usually be feeding in both a question and an answer into an LLM manually. Typically, the context would be retrieved from a vector database, via web search, or from elsewhere. We will cover this use-case in full and build a functional RAG pipeline in a future chapter.

**Few Shot Prompting**

Many State-of-the-Art (SotA) LLMs are incredible at instruction following. Meaning that it requires much less effort to get the intended output or behavior from these models than is the case for older LLMs and smaller LLMs.

Before creating an example let's first see how to use LangChain's few shot prompting objects. We will provide multiple examples and we'll feed them in as sequential human and ai messages so we setup the template like this:

In [11]:
example_prompt= ChatPromptTemplate.from_messages([
    ("human","{input}"),
    ("ai","{output}"),
])#making a basic prompt,human gives the prompt and the ai gives the output, we can use this prompt to test the model with different inputs and outputs

Then we define a list of examples with dictionaries containing the correct input and output keys.

In [12]:
examples = [
    {"input": "Here is query #1", "output": "Here is the answer #1"},
    {"input": "Here is query #2", "output": "Here is the answer #2"},
    {"input": "Here is query #3", "output": "Here is the answer #3"},
]

We then feed both of these into our FewShotChatMessagePromptTemplate object:

In [13]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

print(fewshot_prompt.format())

Human: Here is query #1
AI: Here is the answer #1
Human: Here is query #2
AI: Here is the answer #2
Human: Here is query #3
AI: Here is the answer #3


Using this we can provide different sets of examples or even different individual example_prompt templates to the FewShotChatMessagePromptTemplate object to build our prompt structure. Let's try an real example where we might use few-shot prompting.

we will only modify the eaxmples to the specific topic as example prompt is the same for all.

**Few Shot Example**

Using a tiny LLM limits it's ability, so when asking for specific behaviors or structured outputs it can struggle. For example, we'll ask the LLM to summarize the key points about Aurelio AI using markdown and bullet points. Let's see what happens.

In [14]:
new_system_prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Always answer in markdown format. When doing so please
provide headers, short summaries, follow with bullet
points, then conclude.

Context: {context}
"""

prompt_template.messages[0].prompt.template = new_system_prompt # changing the system prompt template to the new system prompt template

out = pipeline.invoke({"query": query, "context": context}).content 
print(out)

**Overview**

Aurelio AI is an AI company that specializes in developing tooling for AI engineers, particularly in the field of language AI.

**Key Activities**
------------------

* Developing open source frameworks, including:
	+ Semantic Router
	+ Semantic Chunkers
* Providing an AI Platform with tooling to help engineers build with AI
* Offering development services to other organizations to bring their AI tech to market

**Expertise and Focus**
-----------------------

* Strong expertise in building AI agents
* Strong background in information retrieval
* Specialized focus on language AI


We can display our markdown nicely with IPython like so:

In [15]:
from IPython.display import display, Markdown

display(Markdown(out))

**Overview**
===============

Aurelio AI is an AI company that specializes in developing tooling for AI engineers, particularly in the field of language AI.

**Key Activities**
------------------

* Developing open source frameworks, including:
	+ Semantic Router
	+ Semantic Chunkers
* Providing an AI Platform with tooling to help engineers build with AI
* Offering development services to other organizations to bring their AI tech to market

**Expertise and Focus**
-----------------------

* Strong expertise in building AI agents
* Strong background in information retrieval
* Specialized focus on language AI

Let's give an example for our model

In [16]:
examples= [
    {
        "input": "What is the capital of France?",
         "output": (
             "## France\n\n"
            "The capital of France is Paris.\n\n"
            "### Origins\n\n"
            "* The name Paris comes from the Latin word \"Parisini\" which referred to a Celtic people living in the area.\n"
            "* The Romans named the city Lutetia, which means \"the place where the river turns\".\n"
            "* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.\n\n"
            "**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.\n\n"
        )
    },
    {
        "input": "Can you defy gravity?",
        "output": (
                    "## Defying Gravity\n\n"
                    "Whether humans can truly break or escape gravitational pull is a classic scientific inquiry.\n\n"
                    "### The Scientific Reality\n\n"
                    "* From a purely physical standpoint, no, you cannot defy or break the laws of gravity.\n"
                    "* Gravity is a fundamental force of nature caused by mass warping the fabric of spacetime.\n\n"
                    "### Overcoming vs. Defying\n\n"
                    "* While we cannot turn gravity off, we can use other physics principles to counteract it.\n"
                    "* Airplanes use aerodynamic lift, rockets use mechanical thrust, and magnets use electromagnetic repulsion to rise upward.\n"
                    "* These methods match or exceed the downward pull rather than erasing the gravitational field itself.\n\n"
                    "### In True Microgravity\n\n"
                    "* Astronauts floating in orbit appear to completely defy gravity, but they are actually in a constant state of free fall.\n"
                    "* Earth's gravity is still active in orbit, acting as the centripetal force that prevents the space station from flying off into deep space.\n\n"
                    "**To conclude**, while humans have developed incredible technological workarounds to temporarily master and conquer its effects, it remains scientifically impossible to truly switch off or isolate yourself from the laws of gravity.\n"
)
    }
]

We feed these into our fewshot obejct

In [17]:
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt= example_prompt,
    examples=examples,
    )

In [18]:
out = fewshot_prompt.format()

display(Markdown(out))

Human: What is the capital of France?
AI: ## France

The capital of France is Paris.

### Origins

* The name Paris comes from the Latin word "Parisini" which referred to a Celtic people living in the area.
* The Romans named the city Lutetia, which means "the place where the river turns".
* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.

**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.


Human: Can you defy gravity?
AI: ## Defying Gravity

Whether humans can truly break or escape gravitational pull is a classic scientific inquiry.

### The Scientific Reality

* From a purely physical standpoint, no, you cannot defy or break the laws of gravity.
* Gravity is a fundamental force of nature caused by mass warping the fabric of spacetime.

### Overcoming vs. Defying

* While we cannot turn gravity off, we can use other physics principles to counteract it.
* Airplanes use aerodynamic lift, rockets use mechanical thrust, and magnets use electromagnetic repulsion to rise upward.
* These methods match or exceed the downward pull rather than erasing the gravitational field itself.

### In True Microgravity

* Astronauts floating in orbit appear to completely defy gravity, but they are actually in a constant state of free fall.
* Earth's gravity is still active in orbit, acting as the centripetal force that prevents the space station from flying off into deep space.

**To conclude**, while humans have developed incredible technological workarounds to temporarily master and conquer its effects, it remains scientifically impossible to truly switch off or isolate yourself from the laws of gravity.


We then pull all of this together with our system prompt and final user query to create our final prompt and feed it into our LLM.

In [19]:
fewshot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'What is the capital of France?', 'output': '## France\n\nThe capital of France is Paris.\n\n### Origins\n\n* The name Paris comes from the Latin word "Parisini" which referred to a Celtic people living in the area.\n* The Romans named the city Lutetia, which means "the place where the river turns".\n* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.\n\n**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world\'s greatest cultural and economic centres.\n\n'}, {'input': 'Can you defy gravity?', 'output': "## Defying Gravity\n\nWhether humans can truly break or escape gravitational pull is a classic scientific inquiry.\n\n### The Scientific Reality\n\n* From a purely physical standpoint, no, you cannot defy or break the laws of gravity.\n* Gravity is a fundamental force of nature caused by mass warping the fabric of spacetime.\n\n### Overcomi

In [20]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", new_system_prompt),
    fewshot_prompt,#we can also use the fewshot prompt as a message in the prompt template, this will allow us to use the fewshot prompt as a part of the system prompt and we can also use it to provide examples to the model
    ("user", "{query}"),
])

Now feed this back into our pipeline:

In [21]:
pipeline = prompt_template | llm
out = pipeline.invoke({"query": query, "context": context}).content
display(Markdown(out))

## Aurelio AI Overview
### Summary
Aurelio AI is an AI company that specializes in developing tooling for AI engineers.

### Key Focus Areas
* **Language AI**: The team has strong expertise in building AI agents and a background in information retrieval.
* **Open Source Frameworks**: They have developed several open source frameworks, including Semantic Router and Semantic Chunkers.
* **AI Platform**: Aurelio AI provides an AI platform that offers engineers tooling to help them build with AI.

### Additional Services
* **Development Services**: The team also provides development services to other organizations to help bring their AI tech to market.

### Expertise in LangChain Ecosystem
* **LangChain Experts**: In September 2024, Aurelio AI became experts in the LangChain ecosystem after a long track record of delivering AI solutions built with the LangChain ecosystem.

**Chain of Thought Prompting**

We'll take a look at one more commonly used prompting technique called chain of thought (CoT). CoT is a technique that encourages the LLM to think through the problem step by step before providing an answer. The idea being that by breaking down the problem into smaller steps, the LLM is more likely to arrive at the correct answer and we are less likely to see hallucinations.

To implement CoT we don't need any specific LangChain objects, instead we are simply modifying how we instruct our LLM within the system prompt. We will ask the LLM to list the problems that need to be solved, to solve each problem individually, and then to arrive at the final answer.

Let's first test our LLM without CoT prompting.

In [22]:
no_cot_system_prompt = """
Be a helpful assistant and answer the user's question.

You MUST answer the question directly without any other
text or explanation.
"""

no_cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", no_cot_system_prompt),
    ("user", "{query}"),
])

Nowadays most LLMs are trained to use CoT prompting by default, so we actually need to instruct it not to do so for this example which is why we added "You MUST answer the question directly without any other text or explanation." to our system prompt.

In [28]:
query = "How many keystrokes are needed to type every number from 1 to 500 inclusive? Count one keystroke per digit. Do not count anything beyond 500."

no_cot_pipeline = no_cot_prompt_template | llm
no_cot_result = no_cot_pipeline.invoke({"query": query}).content
print(no_cot_result)

2500


The actual answer is 1392, but the LLM without CoT just hallucinates and gives us a guess. Now, we can add explicit CoT prompting to our system prompt to see if we can get a better result.

In [30]:
# Define the chain-of-thought prompt template
cot_system_prompt = """
Be a helpful assistant and answer the user's question.

To answer the question, you must:

- The range is only from 1 to 500 inclusive.
- Do not introduce numbers outside that range.
- Count exactly 1 keystroke per digit typed.
- Show the count for 1-9, 10-99, and 100-500 separately.
- Then add them and give the final answer only once.
- read the question carefully and understand what is being asked.
- List systematically and in precise detail all
  subproblems that need to be solved to answer the
  question.
- Solve each sub problem INDIVIDUALLY and in sequence.
- Finally, use everything you have worked through to
  provide the final answer.
"""

cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", cot_system_prompt),
    ("user", "{query}"),
])

cot_pipeline = cot_prompt_template | llm

In [31]:
cot_result = cot_pipeline.invoke({"query": query}).content
display(Markdown(cot_result))

To solve this problem, we need to break it down into subproblems based on the range of numbers.

**Subproblem 1: Numbers 1-9**
For each number in this range (1-9), there is only one keystroke needed since they are single-digit numbers.
Count = 9 * 1 = 9

**Subproblem 2: Numbers 10-99**
For two-digit numbers, we need to count the keystrokes for both digits. There are 90 such numbers in this range (from 10 to 99).
Count = 90 * 2 = 180

**Subproblem 3: Numbers 100-500**
For three-digit numbers, we need to count the keystrokes for all three digits. There are 401 such numbers in this range (from 100 to 500).
Count = 401 * 3 = 1203

Now, let's add up the counts from each subproblem:

9 + 180 + 1203 = 1392

Therefore, a total of **1,392 keystrokes** are needed to type every number from 1 to 500 inclusive.

Now we get a much better result! Our LLM provides us with a final answer of 1392 which is correct. Finally, as mentioned most LLMs are now trained to use CoT prompting by default. So let's see what happens if we don't explicitly tell the LLM to use CoT.

In [32]:
system_prompt = """
Be a helpful assistant and answer the user's question.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{query}"),
])

pipeline = prompt_template | llm

In [33]:
result = pipeline.invoke({"query": query}).content
display(Markdown(result))

To calculate the total number of keystrokes, we need to consider the numbers with one digit (1-9), two digits (10-99), and three digits (100-499).

For numbers with one digit (1-9), there are 9 numbers, each requiring 1 keystroke. So, that's 9 keystrokes.

For numbers with two digits (10-99), we have 90 numbers (from 10 to 99). Each of these numbers requires 2 keystrokes (one for the tens digit and one for the units digit). Therefore, this group contributes 90 x 2 = 180 keystrokes.

For numbers with three digits (100-499), there are 400 numbers (from 100 to 499). Each of these numbers requires 3 keystrokes (one for the hundreds digit, one for the tens digit, and one for the units digit). Therefore, this group contributes 400 x 3 = 1200 keystrokes.

Now, let's add up all the keystrokes: 9 + 180 + 1200 = 1389 keystrokes.

However, we still need to consider the number 500. It requires 3 keystrokes (one for the hundreds digit, one for the tens digit, and one for the units digit). So, we add 3 more keystrokes to our total: 1389 + 3 = 1392 keystrokes.

Therefore, it takes a total of 1392 keystrokes to type every number from 1 to 500 inclusive.